# Amir's battle prediction experiment

Explore the available battle observations, build 27 numeric features, and
compare linear and tree models, including **XGBoost**, **PCA** and **RFE**.

Use the fixed development folds for comparison. The separate 20% holdout and
submission generation remain off by default. Run from top to bottom; outputs
are saved under `outputs/amirhosein/<run>/`.

In [ ]:
from pathlib import Path
from datetime import datetime
from collections import Counter
import sys
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.feature_selection import RFE, VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").is_file()
             and (p / "src/pokemon_challenge").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this repository folder or run from a notebook inside it.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from pokemon_challenge.experiment import BattleExperiment, Candidate
from pokemon_challenge.paths import data_dir
from pokemon_challenge.data import iter_battles
from pokemon_challenge.validation.splits import make_splits
from pokemon_challenge.features.winning_teams import (
    canonical_team, team_key_features,
)
import xgboost

OWNER = "amirhosein"
RUN = datetime.now().strftime("%Y%m%d_%H%M%S_%f_selected_features")
DATA_DIR = data_dir()  # data/raw/, data/, Kaggle, or POKEMON_DATA_DIR
print("Project:", ROOT)
print("Data:", DATA_DIR)
print("Run:", OWNER, RUN)

RFE_FEATURES = 22
PCA_VARIANCE = 0.95
plt.rcParams.update({'font.size': 11, 'figure.figsize': (9, 4.5),
                     'axes.spines.top': False, 'axes.spines.right': False})
print('XGBoost:', xgboost.__version__)

## 1. Explore the observed data

Use the shared split settings to explore **development rows only**. The outcome
is used here for descriptive comparisons; the holdout is reserved for evaluation.
Only the first 30 timeline entries are available to these summaries.

In [ ]:
profile_rows = []
team_names = {}
for battle in iter_battles(DATA_DIR / 'train.jsonl'):
    roster = battle.get('p1_team_details') or []
    lead = battle.get('p2_lead_details') or {}
    timeline = (battle.get('battle_timeline') or [])[:30]
    key = team_key_features(battle)['team_key']
    team = canonical_team(battle)
    if team:
        if key in team_names and team_names[key] != team:
            raise ValueError('Team-key collision; use a different key representation.')
        team_names[key] = team
    fainted = {}
    for side in ('p1', 'p2'):
        fainted[side] = {state['name'] for turn in timeline
                        if (state := turn.get(f'{side}_pokemon_state') or {}).get('name')
                        and (state.get('hp_pct') == 0 or state.get('status') == 'fnt')}
    profile_rows.append({
        'battle_id': battle['battle_id'], 'won': int(battle['player_won']),
        'team_key': key, 'team_size': len(roster), 'observed_turns': len(timeline),
        'p1_mean_speed': np.mean([pokemon['base_spe'] for pokemon in roster]) if roster else np.nan,
        'p2_lead_speed': lead.get('base_spe', np.nan),
        'real_type_count': len({kind for pokemon in roster for kind in pokemon.get('types', [])
                                if kind != 'notype'}),
        'remaining_count_gap': len(fainted['p2']) - len(fainted['p1']),
    })
profiles = pd.DataFrame(profile_rows)
validation = json.loads((ROOT / 'configs/validation.json').read_text())
eda_development, _, _, _ = make_splits(
    profiles['battle_id'], profiles['won'], seed=validation['seed'],
    n_splits=validation['n_splits'], holdout_size=validation['holdout_size'],
)
exploration = profiles.iloc[eda_development].copy()
print(f'{len(exploration):,} development battles; {exploration.team_key.nunique():,} distinct full teams.')
print('Observed timeline lengths:', sorted(int(value) for value in exploration.observed_turns.unique()))
display(exploration[['team_size', 'real_type_count', 'p1_mean_speed', 'p2_lead_speed']]
        .describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
classes = exploration.won.value_counts().reindex([0, 1], fill_value=0)
bars = axes[0].bar(['P1 loss', 'P1 win'], classes, color=['#8B99A5', '#356A95'])
axes[0].bar_label(bars, padding=3)
axes[0].set(ylabel='Development battles', title='Outcome balance')
axes[0].set_ylim(0, classes.max() * 1.15)
axes[1].hist(exploration.p1_mean_speed, bins=25, color='#356A95', alpha=0.65,
             label='P1 team mean', histtype='stepfilled')
axes[1].hist(exploration.p2_lead_speed, bins=25, color='#AE722C',
             histtype='step', linewidth=2, label='P2 lead')
axes[1].set(xlabel='Base speed', ylabel='Development battles', title='Available speed information')
axes[1].legend(frameon=False)
fig.tight_layout()
plt.show()
print(f'Development P1 win rate: {exploration.won.mean():.1%}.')
print('Team means summarize a roster; P2 lead stats describe only one Pokémon.')

In [ ]:
progress = exploration.groupby('remaining_count_gap').won.agg(battles='size', win_rate='mean')
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(progress.index.astype(str), progress.win_rate, color='#356A95')
ax.bar_label(bars, labels=[f'n={count:,}' for count in progress.battles], padding=3, fontsize=9)
ax.axhline(0.5, color='#666666', linestyle='--', linewidth=1)
ax.set(xlabel='P1 minus P2 presumed remaining Pokémon', ylabel='Observed P1 win rate',
       ylim=(0, 1.12), title='Observed fainting and outcome · development data')
ax.yaxis.set_major_formatter(PercentFormatter(1))
fig.tight_layout()
plt.show()
display(progress.round(3))

### From exploration to feature choices

- The target balance supplies the majority-class reference for model comparison.
- Team composition varies, so summarize all six base stats and real type coverage.
  P2 lead information is available even though its full roster is hidden.
- Compare observed fainting with the outcome to inspect battle progress. Use
  remaining-count differences, move counts and switches alongside team features.
- HP changes, healing and boost summaries describe the observed battle dynamics.
  Their assumptions are defined below; they do not identify causal effects.

These checks motivate a compact feature set. Cross-validation tests its predictive
value; the plots alone do not establish which features improve a model.

### The most frequently winning full teams

A team is its six species, independent of roster order, moves or items. Rank by
**number of wins**, breaking ties by win rate and then a stable team key. Frequent
teams have more opportunities to win, so show battles and win rate alongside wins.
This exploratory table is descriptive; team identity and winning-team flags are
excluded from the model inputs.

In [ ]:
winning_teams = exploration.groupby('team_key').won.agg(
    battles='size', wins='sum', win_rate='mean'
).reset_index()
winning_teams = winning_teams[winning_teams.wins > 0].sort_values(
    ['wins', 'win_rate', 'team_key'], ascending=[False, False, True]
).reset_index(drop=True)
winning_teams['team'] = winning_teams.team_key.map(lambda key: ', '.join(team_names[key]))
winning_teams.index = winning_teams.index + 1
winning_teams.index.name = 'rank'
with pd.option_context('display.max_colwidth', None, 'display.max_columns', None):
    display(winning_teams[['team', 'battles', 'wins', 'win_rate']].head(6).round(3))
for top_n in (2, 5, 6):
    coverage = exploration.team_key.isin(winning_teams.head(top_n).team_key).mean()
    print(f'Top {top_n} teams cover {coverage:.1%} of development battles.')

## 2. Core features and assumptions

There are **27 numeric inputs**: the previous 24-input set with starting P2
lead speed replaced by four observed-roster base-stat averages.
All seven candidates use exactly this set.
Selection used development labels, so subsequent CV scores are exploratory;
the separate holdout remains reserved for final evaluation. Winning-team flags
and individual elemental-type counts remain excluded.

| Features | Definition |
| --- | --- |
| `p1_mean_{hp,atk,def,spe}` | Mean base stat over P1's roster |
| `p2_mean_{hp,atk,def,spe}` | Mean base stats across distinct P2 species observed by turn 30, including the starting lead |
| `p1_alive_at_turn30`, `alive_diff_at_turn30` | P1 presumed remaining count; P1 minus P2 remaining count |
| `p1_mean_boosts`, `p2_mean_boosts` | Mean across turns of the sum of five boost stages |
| `p2_no_of_moves` | Number of non-null recorded P2 moves |
| `p1_dmg`, `p2_dmg` | Positive observed opponent HP drops, summed by Pokémon |
| `p1_heal`, `p2_heal` | Positive observed own HP increases, summed by Pokémon |
| `p1_switch` | Consecutive P1 active-name changes, including forced replacements |
| `p1_boost_sum_at_turn30`, `p2_boost_sum_at_turn30` | Net sum of five active stat stages at turn 30 |
| `p1_no_damage_moves`, `p2_no_damage_moves` | Recorded moves with no net opponent HP decrease in comparable adjacent observations |
| `p1_team_hp_pct_at_turn30`, `p2_team_hp_pct_at_turn30` | Latest observed team HP as a percentage of six full HP bars; unseen members are full |
| `p1_special_moves` | Recorded P1 moves in the `special` category |

Final boost sums use the active Pokémon at turn 30. Adjacent HP comparisons
skip the first observation, switches and missing states. Null moves do not
count as zero-damage moves. HP changes include residual effects and healing
can mask damage. Latest team HP can be stale for benched members; each member
contributes equally. Helper calculations may track additional quantities,
but the returned model inputs are restricted to `FEATURE_COLUMNS`.

### Observed P2 roster

Collect the starting lead and distinct P2 species in the first 30 timeline entries
with turn numbers 1–30. Each species contributes once, even after repeated switches.
Average base HP, Attack, Defense and Speed using the static species lookup
from supplied roster/lead records; no labels or level scaling are used.
Unseen team members are excluded. Unknown species raise an error.

### Equally weighted damage surplus

For each turn 1–30, track cumulative damage **received** minus healing by
summing each Pokémon's observed HP losses and subtracting its HP gains.
P1's offensive curve is P2's net damage received; P2's curve is P1's net
damage received. Thus healing is subtracted from the damage to the **same team**.
An advantage for P1 means its offensive curve is higher.

Let `gap(t) = P1_net_damage_dealt(t) - P2_net_damage_dealt(t)`.
The two additional model inputs are:

- `p1_damage_surplus = sum(max(gap(t), 0), t=1..30) / 30`
- `p2_damage_surplus = sum(max(-gap(t), 0), t=1..30) / 30`

These are nonnegative average advantages across the 30 turns, measured in
full-Pokémon HP bars. Every turn has equal weight. Each end-of-turn value occupies one unit-width bin;
there is no interpolation between turns. Missing rounds carry the latest
cumulative values forward, with zero before the first observation. Only the
first 30 timeline entries with turn numbers in 1–30 are used.

A newly observed Pokémon starts at assumed full HP, matching the existing
damage/healing summaries. A switch compares the returning Pokémon to its
own previous HP, rather than the outgoing Pokémon. HP changes remain observed
net effects, including residual damage; they are not exact move attribution.

### Key assumptions

All timeline features stop after the first 30 entries. Fainted means zero HP or
status `fnt`, with each name counted once. Sleep and freeze do not imply fainting;
unseen members are presumed remaining in a six-member team.

Damage/healing compare the same Pokémon's successive observations, including
returns after switching. A newly seen Pokémon starts from an assumed HP of 1.0,
as an initial-health proxy. These totals include changes
from unobserved intervals and do not prove which move caused them. HP units are
fractions of a Pokémon's maximum, not absolute HP points.

### Shared feature interface

`extract_selected_features` takes one battle and returns the same 27 numeric keys
for every observation. The shared runner removes the ID and target before calling
it and prefixes its output columns with `selected__`. No learned statistics or
outcome-derived rates are computed here.


In [ ]:
from pokemon_challenge.features.observed_roster import p2_observed_roster_stats

TEAM_SIZE = 6
MAX_OBSERVED_TURNS = 30

STATS = ('hp', 'atk', 'def', 'spa', 'spd', 'spe')
TYPES = (
    'bug', 'dark', 'dragon', 'electric', 'fighting', 'fire', 'flying',
    'ghost', 'grass', 'ground', 'ice', 'normal', 'poison', 'psychic',
    'rock', 'steel', 'water',
)
BOOST_STATS = ('atk', 'def', 'spa', 'spd', 'spe')
# Retain features through p1_mean_def in descending absolute coefficient order.
FEATURE_COLUMNS = (
    'p2_team_hp_pct_at_turn30',
    'p1_dmg',
    'p1_team_hp_pct_at_turn30',
    'p2_dmg',
    'p2_heal',
    'p1_heal',
    'p1_boost_sum_at_turn30',
    'p1_no_damage_moves',
    'p1_mean_hp',
    'alive_diff_at_turn30',
    'p1_alive_at_turn30',
    'p2_no_damage_moves',
    'p2_boost_sum_at_turn30',
    'p2_mean_boosts',
    'p1_mean_boosts',
    'p1_mean_spe',
    'p1_switch',
    'p2_no_of_moves',
    'p1_special_moves',
    'p1_mean_atk',
    'p2_mean_hp',
    'p2_mean_atk',
    'p2_mean_def',
    'p2_mean_spe',
    'p1_mean_def',
    'p1_damage_surplus',
    'p2_damage_surplus',
)
REQUESTED_FEATURE_SUFFIXES = (
    "positive_boosts_at_turn30", "boost_sum_at_turn30", "no_damage_moves",
    "fainting_turns", "team_hp_pct_at_turn30", "special_moves",
)
assert len(FEATURE_COLUMNS) == len(set(FEATURE_COLUMNS)) == 27


def extract_requested_features(battle):
    """Summarize the requested final states and observed move outcomes."""
    turns = [turn for turn in (battle.get('battle_timeline') or [])
             if 1 <= turn.get('turn', 0) <= MAX_OBSERVED_TURNS]
    final_turn = next((turn for turn in reversed(turns)
                       if turn['turn'] == MAX_OBSERVED_TURNS), {})
    latest_hp = {
        'p1': {pokemon['name']: 1.0 for pokemon in (battle.get('p1_team_details') or [])},
        'p2': {},
    }
    lead_name = (battle.get('p2_lead_details') or {}).get('name')
    if lead_name:
        latest_hp['p2'][lead_name] = 1.0
    previous_state = {'p1': None, 'p2': None}
    fainted = {'p1': set(), 'p2': set()}
    features = {}
    for side in ('p1', 'p2'):
        for suffix in ('no_damage_moves', 'max_observed_move_damage',
                       'fainting_turns', 'special_moves'):
            features[f'{side}_{suffix}'] = 0.0

    for turn in turns:
        for side in ('p1', 'p2'):
            opponent = 'p2' if side == 'p1' else 'p1'
            state = turn.get(f'{side}_pokemon_state') or {}
            name, hp = state.get('name'), state.get('hp_pct')
            is_fainted = hp == 0 or state.get('status') == 'fnt'
            if name and is_fainted and name not in fainted[side]:
                features[f'{side}_fainting_turns'] += 1
                fainted[side].add(name)
            if name and hp is not None:
                latest_hp[side][name] = 0.0 if is_fainted else float(hp)

            move = turn.get(f'{side}_move_details')
            if move is not None:
                features[f'{side}_special_moves'] += move.get('category') == 'special'
                target = turn.get(f'{opponent}_pokemon_state') or {}
                previous = previous_state[opponent]
                # Only adjacent observations of the same target are comparable.
                if (previous and target.get('name')
                        and previous.get('name') == target['name']
                        and previous.get('hp_pct') is not None
                        and target.get('hp_pct') is not None):
                    drop = max(float(previous['hp_pct']) - float(target['hp_pct']), 0.0)
                    features[f'{side}_no_damage_moves'] += drop == 0.0
                    features[f'{side}_max_observed_move_damage'] = max(
                        features[f'{side}_max_observed_move_damage'], drop)
        # Update both sides together so results do not depend on loop order.
        previous_state = {
            side: turn.get(f'{side}_pokemon_state') or {}
            for side in ('p1', 'p2')
        }

    for side in ('p1', 'p2'):
        state = final_turn.get(f'{side}_pokemon_state') or {}
        boosts = state.get('boosts') or {}
        stages = [float(boosts.get(stat, 0)) for stat in BOOST_STATS]
        features[f'{side}_positive_boosts_at_turn30'] = sum(max(value, 0.0) for value in stages)
        features[f'{side}_negative_boosts_at_turn30'] = sum(min(value, 0.0) for value in stages)
        features[f'{side}_boost_sum_at_turn30'] = sum(stages)
        unseen = max(0, TEAM_SIZE - len(latest_hp[side]))
        features[f'{side}_team_hp_pct_at_turn30'] = (
            100.0 * (sum(latest_hp[side].values()) + unseen) / TEAM_SIZE)
    return features


def damage_surplus_curves(battle):
    """End-of-turn cumulative net damage on a fixed, equally weighted 30-turn grid."""
    by_turn = {
        entry.get('turn', index): entry
        for index, entry in enumerate(
            (battle.get('battle_timeline') or [])[:MAX_OBSERVED_TURNS], start=1)
        if 1 <= entry.get('turn', index) <= MAX_OBSERVED_TURNS
    }
    previous_hp = {'p1': {}, 'p2': {}}
    net_received = {'p1': 0.0, 'p2': 0.0}
    curves = {'p1_net_damage_dealt': [], 'p2_net_damage_dealt': []}
    for turn_number in range(1, MAX_OBSERVED_TURNS + 1):
        entry = by_turn.get(turn_number, {})
        for side in ('p1', 'p2'):
            state = entry.get(f'{side}_pokemon_state') or {}
            name, hp = state.get('name'), state.get('hp_pct')
            if name and hp is not None:
                hp = float(hp)
                # Signed drop equals observed damage received minus own healing.
                net_received[side] += previous_hp[side].get(name, 1.0) - hp
                previous_hp[side][name] = hp
        curves['p1_net_damage_dealt'].append(net_received['p2'])
        curves['p2_net_damage_dealt'].append(net_received['p1'])
    curves = {key: np.asarray(values, dtype=float) for key, values in curves.items()}
    curves['turn'] = np.arange(1, MAX_OBSERVED_TURNS + 1)
    return curves


def extract_damage_surplus_features(battle):
    curves = damage_surplus_curves(battle)
    gap = curves['p1_net_damage_dealt'] - curves['p2_net_damage_dealt']
    return {
        'p1_damage_surplus': float(np.maximum(gap, 0).mean()),
        'p2_damage_surplus': float(np.maximum(-gap, 0).mean()),
    }


def extract_selected_features(battle):
    roster = battle.get('p1_team_details') or []
    lead = battle.get('p2_lead_details') or {}
    timeline = (battle.get('battle_timeline') or [])[:MAX_OBSERVED_TURNS]
    features = {}
    for stat in STATS:
        features[f'p1_mean_{stat}'] = float(np.mean([
            pokemon.get(f'base_{stat}', 0) for pokemon in roster
        ])) if roster else 0.0
    type_counts = Counter(
        str(kind).lower()
        for pokemon in roster
        for kind in pokemon.get('types', [])
        if str(kind).lower() != 'notype'
    )
    unknown_types = set(type_counts) - set(TYPES)
    if unknown_types:
        raise ValueError(f'Unknown real roster types: {sorted(unknown_types)}')
    features['p1_team_diversity'] = len(type_counts)
    for stat in STATS:
        features[f'p2_lead_{stat}'] = lead.get(f'base_{stat}', 0)

    fainted = {'p1': set(), 'p2': set()}
    previous_hp = {'p1': {}, 'p2': {}}
    previous_name = {'p1': None, 'p2': None}
    boost_sums = {'p1': [], 'p2': []}
    for side in ('p1', 'p2'):
        features[f'{side}_no_of_moves'] = 0
        features[f'{side}_dmg'] = 0.0
        features[f'{side}_heal'] = 0.0
        features[f'{side}_switch'] = 0

    for turn in timeline:
        for side in ('p1', 'p2'):
            opponent = 'p2' if side == 'p1' else 'p1'
            state = turn.get(f'{side}_pokemon_state') or {}
            name = state.get('name')
            boosts = state.get('boosts') or {}
            boost_sums[side].append(sum(boosts.get(stat, 0) for stat in BOOST_STATS))
            features[f'{side}_no_of_moves'] += turn.get(f'{side}_move_details') is not None
            if not name:
                previous_name[side] = None
                continue
            if previous_name[side] is not None and previous_name[side] != name:
                features[f'{side}_switch'] += 1
            previous_name[side] = name
            hp = state.get('hp_pct')
            if hp == 0 or state.get('status') == 'fnt':
                fainted[side].add(name)
            if hp is not None:
                hp = float(hp)
                old_hp = previous_hp[side].get(name, 1.0)
                features[f'{opponent}_dmg'] += max(old_hp - hp, 0.0)
                features[f'{side}_heal'] += max(hp - old_hp, 0.0)
                previous_hp[side][name] = hp

    for side in ('p1', 'p2'):
        features[f'{side}_alive_at_turn30'] = TEAM_SIZE - len(fainted[side])
        features[f'{side}_mean_boosts'] = float(np.mean(boost_sums[side])) if boost_sums[side] else 0.0
    features['alive_diff_at_turn30'] = features['p1_alive_at_turn30'] - features['p2_alive_at_turn30']
    features.update(p2_observed_roster_stats(battle))
    features.update(extract_requested_features(battle))
    features.update(extract_damage_surplus_features(battle))
    return {name: features[name] for name in FEATURE_COLUMNS}



FEATURE_FUNCTIONS = {"selected": extract_selected_features}
print("Selected numeric inputs:", len(FEATURE_COLUMNS))



### PCA and RFE inside training folds

PCA and RFE are fitted separately within each training fold. Team lookup keys
and winning-team membership flags are excluded from every candidate.


In [ ]:
class AmirExperiment(BattleExperiment):
    def _pipeline(self, candidate):
        if candidate.model not in {'pca_logistic', 'rfe_logistic', 'rfe_pca_logistic'}:
            return super()._pipeline(candidate)
        seed = self.validation['seed']
        steps = [
            ('imputer', SimpleImputer(strategy='median', keep_empty_features=True)),
            ('constant_filter', VarianceThreshold()),
            ('scaler', StandardScaler()),
        ]
        if 'rfe' in candidate.model:
            steps.append(('rfe', RFE(
                LogisticRegression(C=1.0, max_iter=2000, random_state=seed),
                n_features_to_select=RFE_FEATURES, step=0.1,
            )))
        if 'pca' in candidate.model:
            steps.append(('pca', PCA(n_components=PCA_VARIANCE, svd_solver='full')))
        steps.append(('model', LogisticRegression(
            random_state=seed, **{'C': 1.0, 'max_iter': 2000, **candidate.params}
        )))
        return Pipeline(steps)

    def compare(self, candidates):
        results = super().compare(candidates)
        for candidate in candidates:
            if 'pca' in candidate.model:
                path = self.output_path / f'selection_frequency_{candidate.name}.csv'
                frequency = pd.read_csv(path)
                frequency['selected_in_folds'] = np.nan
                frequency['note'] = 'Not applicable: PCA transforms inputs into components.'
                frequency.to_csv(path, index=False)
        return results

## 3. Load features and create fixed validation splits

The shared path helper locates the official inputs. `configs/validation.json`
reserves 20% as a holdout and uses five stratified development folds, with seed 42.
Every candidate uses identical assignments. Test data is read only for submission.

The shared splitter does not group duplicate or related battles, which can make
scores optimistic if these cross partitions. Rerun setup for a new run folder.

In [ ]:
experiment = AmirExperiment(
    owner=OWNER,
    run=RUN,
    feature_functions=FEATURE_FUNCTIONS,
    data_path=DATA_DIR,
)
assert experiment.X.shape[1] == len(FEATURE_COLUMNS) == 27
assert set(experiment.X.columns) == {f"selected__{name}" for name in FEATURE_COLUMNS}
assert np.array_equal(experiment.development, eda_development)
if not np.isfinite(experiment.X.to_numpy()).all():
    raise ValueError("All selected training features must be finite numeric values.")

(experiment.output_path / 'notebook_settings.json').write_text(json.dumps({
    'rfe_features': RFE_FEATURES, 'pca_variance': PCA_VARIANCE,
    'core_features': len(FEATURE_COLUMNS),
    'p2_roster': 'distinct species including lead through turn 30; unscaled base stats; unseen excluded',
    'requested_features': [name for name in FEATURE_COLUMNS
                           if any(name == f'{side}_{suffix}'
                                  for side in ('p1', 'p2')
                                  for suffix in REQUESTED_FEATURE_SUFFIXES)],
    'damage_surplus': {
        'features': ['p1_damage_surplus', 'p2_damage_surplus'],
        'weight': 'equal for all turns 1 through 30',
        'normalization': 30, 'integration': 'end-of-turn unit-width rectangles',
        'net_damage_dealt': 'opponent cumulative observed damage minus opponent healing',
        'missing_turns': 'carry forward', 'new_pokemon_hp': 1.0,
    },
    'move_damage': 'adjacent same-target net HP loss; recorded moves only',
    'negative_boosts': 'signed sum',
    'team_hp': 'latest observed HP; unseen full; equal member weighting; 0-100 percent', 'winning_team_flags': [],
    'xgboost_version': xgboost.__version__,
}, indent=2) + '\n')
print("Core predictors: 27; winning-team flags: 0.")
display(experiment.assignments.groupby(["partition", "validation_fold"])
        .size().rename("battles").to_frame())


### Visualize the two damage surpluses

This development battle is the first with positive surplus for both sides
(or the first development battle if none exists). Its outcome is not used
to select it. Change `example_id` to inspect another development battle.
The chart shows cumulative net damage dealt. Shading marks the leading
player's surplus with equal weight for every turn. Each side's shaded area
divided by 30 equals its feature value.


In [ ]:
development_features = experiment.X.iloc[experiment.development]
both_lead = ((development_features['selected__p1_damage_surplus'] > 0)
             & (development_features['selected__p2_damage_surplus'] > 0))
example_index = (development_features.index[both_lead][0] if both_lead.any()
                 else development_features.index[0])
example_id = experiment.ids[example_index]
assert example_id in set(experiment.ids[experiment.development])
example_battle = next(battle for battle in iter_battles(DATA_DIR / 'train.jsonl')
                      if battle['battle_id'] == example_id)
curves = damage_surplus_curves(example_battle)
surplus = extract_damage_surplus_features(example_battle)
edges = np.arange(MAX_OBSERVED_TURNS + 1)
p1_curve, p2_curve = curves['p1_net_damage_dealt'], curves['p2_net_damage_dealt']
fig, ax = plt.subplots(figsize=(11, 5))
p1, p2 = p1_curve, p2_curve
p1_steps, p2_steps = np.r_[p1, p1[-1]], np.r_[p2, p2[-1]]
ax.step(edges, p1_steps, where='post', color='#356A95', linewidth=2,
        label='P1 net damage dealt')
ax.step(edges, p2_steps, where='post', color='#AE722C', linewidth=2,
        linestyle='--', label='P2 net damage dealt')
ax.fill_between(edges, p2_steps, np.maximum(p1_steps, p2_steps),
                step='post', color='#356A95', alpha=.25, label='P1 surplus')
ax.fill_between(edges, p1_steps, np.maximum(p1_steps, p2_steps),
                step='post', color='#AE722C', alpha=.25, hatch='//',
                label='P2 surplus')
ax.axhline(0, color='#666666', linewidth=.8)
ax.set(title='Cumulative damage dealt minus opponent healing · equal turn weights', ylabel='HP bars')
ax.legend(frameon=False, loc='upper left', ncol=2, fontsize=9)
ax.set(xlabel='Observed turn (each bin ends at its turn number)',
             xlim=(0, MAX_OBSERVED_TURNS), xticks=np.arange(0, 31, 5))
fig.suptitle(f'Damage surplus · development battle {example_id}', fontsize=13)
fig.tight_layout(rect=(0, .05, 1, .96))
fig.text(.5, .02, f"P1 damage surplus: {surplus['p1_damage_surplus']:.4f} HP bars"
         f"     |     P2 damage surplus: {surplus['p2_damage_surplus']:.4f} HP bars",
         ha='center', fontsize=11)
fig.savefig(experiment.output_path / 'damage_surplus_example.png', dpi=150,
            bbox_inches='tight')
plt.show()


### Check the engineered features against the outcome

Inspect the strongest single-feature associations on development data. This
connects the observed battle summaries to the chosen features and can expose
constants or unexpected values. Pearson correlation with a binary outcome is a
univariate association: it misses nonlinear effects and does not establish
causality or unique value among correlated predictors. RFE still learns its
selection separately within training folds.

In [ ]:
core_columns = [f'selected__{name}' for name in FEATURE_COLUMNS]
core_development = experiment.X.iloc[experiment.development][core_columns]
correlations = core_development.corrwith(experiment.y.iloc[experiment.development]).dropna()
top_associations = correlations.loc[correlations.abs().nlargest(12).index].sort_values()
fig, ax = plt.subplots(figsize=(9, 5))
labels = [name.removeprefix('selected__') for name in top_associations.index]
ax.barh(labels, top_associations, color='#356A95')
ax.axvline(0, color='#666666', linewidth=1)
ax.set(xlabel='Pearson correlation with P1 win (development rows)',
       title='Core feature checks · 12 strongest absolute associations')
fig.tight_layout()
plt.show()
print('Missing core feature values:', int(core_development.isna().sum().sum()))
print('Constant core features:', [name.removeprefix('selected__')
                                  for name in core_development.columns
                                  if core_development[name].nunique() <= 1])

## 4. Compare models

Compare seven models on the 27 retained and requested features, including XGBoost and
the PCA/RFE variants. All candidates use identical development folds, and the
holdout is excluded.

The comparison reports raw input columns and retained model dimensions.
Constants can be removed. PCA retains 95% of scaled training variance; RFE retains
22 original predictors. Fold standard deviation is not a confidence interval.


In [ ]:
XGB_PARAMS = {
    'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.04,
    'subsample': 0.8, 'colsample_bytree': 0.8,
    'min_child_weight': 5, 'reg_lambda': 5.0,
}
candidates = [
    Candidate('selected_xgboost', groups=('selected',), model='xgboost', params=XGB_PARAMS),
    Candidate('pca_logistic', groups=('selected',), model='pca_logistic'),
    Candidate('rfe_logistic', groups=('selected',), model='rfe_logistic'),
    Candidate('rfe_pca_logistic', groups=('selected',), model='rfe_pca_logistic'),
    Candidate('selected_forest', groups=('selected',), model='random_forest',
              params={'n_estimators': 200, 'min_samples_leaf': 5}),
    Candidate(
        "selected_logistic",
        groups=("selected",),
        model="logistic_regression",
        percentile=100,
        params={"C": 1.0, "max_iter": 2000},
    ),
    Candidate(
        "selected_hist_boosting",
        groups=("selected",),
        model="hist_gradient_boosting",
        percentile=100,
        params={
            "max_iter": 250,
            "learning_rate": 0.06,
            "max_leaf_nodes": 15,
            "min_samples_leaf": 25,
            "l2_regularization": 5.0,
            "early_stopping": False,
        },
    ),
]
results = experiment.compare(candidates)
display(results[['candidate', 'input_features', 'mean_selected_features',
                 'cv_accuracy', 'fold_std', 'mean_fit_seconds']].round(4))
print("Highest mean CV accuracy:", experiment.best_candidate)


## 5. Inspect validation and reduction

Whiskers show one fold standard deviation. The dashed line is the majority-class
reference on the same folds. RFE selection frequency measures stability across
training folds, not causal importance. PCA creates combinations of inputs, so
original-feature selection frequency is not applicable to its components.

The PCA diagnostic below is fitted on development data only for interpretation;
it does not provide an additional validation score or inspect the holdout.

In [ ]:
ranked = results.sort_values('cv_accuracy')
fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.barh(ranked['candidate'], ranked['cv_accuracy'], color='#356A95')
ax.errorbar(ranked['cv_accuracy'], ranked['candidate'], xerr=ranked['fold_std'],
            fmt='none', ecolor='#263746', capsize=3)
ax.axvline(results['dummy_accuracy'].iloc[0], color='#666666', linestyle='--',
           label='Majority-class reference')
ax.bar_label(bars, labels=[f'{value:.2%}' for value in ranked['cv_accuracy']], padding=12)
ax.set(xlim=(0, 1), xlabel='Development CV accuracy', title='Model comparison · five fixed folds')
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.legend(loc='lower left', frameon=False)
fig.tight_layout()
plt.show()

In [ ]:
pca_candidate = next(candidate for candidate in candidates if candidate.name == 'pca_logistic')
pca_model = experiment._pipeline(pca_candidate)
pca_columns = experiment._columns(pca_candidate)
pca_model.fit(experiment.X.iloc[experiment.development][pca_columns], experiment.y.iloc[experiment.development])
pca = pca_model.named_steps['pca']
cumulative = np.cumsum(pca.explained_variance_ratio_)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.arange(1, len(cumulative) + 1), cumulative, color='#356A95', linewidth=2)
ax.axhline(PCA_VARIANCE, color='#666666', linestyle='--', label='95% variance')
ax.set(xlabel='Retained principal components', ylabel='Cumulative explained variance',
       ylim=(0, 1.02), title='PCA compression · development data')
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.legend(frameon=False)
fig.tight_layout()
plt.show()
print(f'PCA retained {pca.n_components_} components from {len(FEATURE_COLUMNS)} inputs.')

rfe_selection = pd.read_csv(experiment.output_path / 'selection_frequency_rfe_logistic.csv')
rfe_selection['feature'] = rfe_selection['feature'].str.removeprefix('selected__')
display(rfe_selection.sort_values(['selected_in_folds', 'feature'],
                                 ascending=[False, True]).head(12))

## 6. Final holdout evaluation — explicit decision

Leave this **False** while choosing features and models. Set it to **True** only when you have chosen your approach from CV. The chosen pipeline is fitted on the 80% development subset and evaluated on the untouched 20% holdout.

Opening the holdout locks further comparisons in this experiment. Do not tune on its score or repeatedly create new runs to work around the lock; doing so would turn the holdout into another tuning set. The guard is per Python experiment object, not a cross-team access control system.

In [ ]:
EVALUATE_HOLDOUT = False
CHOSEN_CANDIDATE = experiment.best_candidate  # or a specific compared candidate name

if EVALUATE_HOLDOUT:
    holdout = experiment.evaluate_holdout(CHOSEN_CANDIDATE)
    print("Candidate:", holdout["candidate"])
    print("Holdout accuracy:", holdout["accuracy"])
    display(pd.DataFrame(holdout["confusion_matrix"],
                         index=["actual loss", "actual win"], columns=["predicted loss", "predicted win"]))
    display(pd.DataFrame(holdout["classification_report"]).T)
    print("Selected features:", holdout["selected_features"])
else:
    print("Holdout remains unopened. Continue comparing using CV only.")


## 7. Train on all labels and generate a submission (optional)

After evaluating the chosen candidate, set `MAKE_SUBMISSION=True`. The shared
runner refits the locked configuration on all labelled training battles, including
the former holdout, and predicts `test.jsonl`. Its labels are unknown locally.

The sample submission supplies the IDs and row order. Files are saved as
`submission.csv`, `model.joblib`, and `final_fit.json` inside this run's output
folder; nothing is uploaded automatically. The saved pipeline accepts the recorded
numeric input columns, so retain this notebook for raw-battle inference.


In [ ]:
MAKE_SUBMISSION = False
if MAKE_SUBMISSION:
    submission = experiment.make_submission()
    display(submission.head())
    print("Upload manually:", experiment.output_path / "submission.csv")
else:
    print("Submission generation skipped.")


## 8. Artifacts and experiment notes

The unique ignored run folder contains:

- `experiment.json`: validation settings, data/split hashes, versions, code commit and dirty-tree status.
- `feature_sources.json`: feature function source where available.
- `splits.csv`: saved holdout and development-fold assignments.
- `candidates.json`, `cv_results.csv`, and `selection_frequency_*.csv`: configurations and CV evidence.
- After holdout evaluation: metrics, selected features, and battle-level predictions.
- After final fitting: `model.joblib`, `submission.csv`, and `final_fit.json`.

Record the hypothesis, interpretation, and exact run/commit in your notes. Uncommitted changes are flagged in the manifest; commit the implementation and retain this notebook/configuration before relying on a final result. Saved source snippets help auditing but do not automatically package custom imports or globals.

Only load model files you trust. Saved outputs are local and ignored by Git; commit the notebook/shared code separately and preserve final artifacts with your team.

Reducer and feature settings are saved in `notebook_settings.json`. Retain this notebook
with any fitted model because it also defines the PCA/RFE runner extension.


In [ ]:
print('Run folder:', experiment.output_path)
print('CV comparison:', experiment.output_path / 'cv_results.csv')